# 🚀 CodeFormer Face Restoration & Enhancer — Google Colab GPU Training

This notebook provides automated end-to-end GPU training for the **Custom AI Enhancer** model:
- **Stage III CFT Fine-Tuning** with **ArcFace Identity Loss** (Iter 2,002 → 20,000)
- **Stage II Transformer Fine-Tuning** for enhanced portrait quality
- **Automatic Google Drive Checkpoint Synchronization**
- **Benchmark Evaluation & Static INT8 ONNX Export**

## 1. System & GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 or A100 GPU.")

## 2. Google Drive Mounting & Checkpoint Backup
Mount Google Drive to persist trained checkpoints across Colab session disconnections.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
backup_dir = '/content/drive/MyDrive/CodeFormer_Experiments'
os.makedirs(backup_dir, exist_ok=True)
print(f"✅ Checkpoints will be automatically backed up to: {backup_dir}")

## 3. Clone Repository & Setup Dependencies

In [ ]:
%cd /content
!git clone https://github.com/supli6669/Enhance-Image.git custom-ai-enhancer
%cd /content/custom-ai-enhancer

# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q facexlib lpips gdown onnx onnxruntime-gpu pyyaml opencv-python scikit-image

# Patch and install BasicSR
!python tools/patch_and_install_basicsr.py

# Link experiment directory to Google Drive for real-time backup
!mkdir -p models/CodeFormer/experiments
!ln -sfn /content/drive/MyDrive/CodeFormer_Experiments models/CodeFormer/experiments/backup
print("✅ Environment configured successfully!")

## 4. Download Model Weights & Datasets

In [ ]:
# Download pre-trained weights
!python tools/download_weights.py

# Verify ArcFace and VQGAN weights exist
import os
print("Checking required model weights:")
weights = [
    'weights/CodeFormer/codeformer.pth',
    'weights/facelib/recognition_arcface_ir_se50.pth',
    'weights/facelib/vqgan_code1024.pth'
]
for w in weights:
    exists = os.path.exists(w)
    status = "✅ Found" if exists else "❌ Missing"
    print(f"  - {w}: {status}")

## 5. Phase 2 & 3: Run Stage III CFT Fine-Tuning (with ArcFace Identity Loss)
Huấn luyện tiếp tục từ checkpoint iter 2,002 → 20,000 với loss nhận diện khuôn mặt ArcFace.

In [ ]:
!python train_custom.py

## 6. Phase 6: Run Stage II Transformer Fine-Tuning
Huấn luyện mô hình Transformer đối chiếu codebook với cấu hình `CodeFormer_stage2_custom.yml`.

In [ ]:
!python -m torch.distributed.run --nproc_per_node=1 models/CodeFormer/basicsr/train.py -opt models/CodeFormer/options/CodeFormer_stage2_custom.yml --launcher pytorch

## 7. Run Quantitative Benchmark Evaluation
Đánh giá định lượng trên 500 mẫu chân dung độc lập (PSNR, SSIM, LPIPS, ArcFace Cosine Identity Similarity, Latency).

In [ ]:
!python tools/evaluate_restoration.py --manifest benchmarks/manifest.csv --device cuda --save-images benchmarks/outputs/colab_gpu/

## 8. Export to ONNX & Static INT8 Quantization
Xuất checkpoint đã huấn luyện sang ONNX Static INT8 tối ưu hóa cho môi trường CPU/GPU.

In [ ]:
!python tools/export_onnx.py --model codeformer --input-pth weights/CodeFormer/codeformer.pth --output-onnx weights/CodeFormer/codeformer.onnx
!python tools/quantize_onnx_static.py
print("🎉 Export and Static INT8 Quantization completed successfully!")